In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/devalshah04/autiz-test-audio/test_clip.m4a
/kaggle/input/datasets/devalshah04/autiz-test-audio/test_clip_expressive.m4a


In [2]:
# Cell 1 — Install all required libraries and prepare environment (run every session)
!pip install faster-whisper opensmile transformers torch jiwer -q
!apt-get install -y mediainfo ffmpeg -q
print("All libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 22.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Read

In [3]:
from faster_whisper import WhisperModel
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("faster-whisper imported successfully")

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
faster-whisper imported successfully


In [4]:
import os
path = "/kaggle/input/datasets/devalshah04/autiz-test-audio/"
print(os.listdir(path))

['test_clip.m4a', 'test_clip_expressive.m4a']


In [5]:
# Cell 2 — Convert both audio clips from M4A to 16kHz mono WAV format

!ffmpeg -i "/kaggle/input/datasets/devalshah04/autiz-test-audio/test_clip.m4a" -ar 16000 -ac 1 "/kaggle/working/test_clip.wav" -y
!ffmpeg -i "/kaggle/input/datasets/devalshah04/autiz-test-audio/test_clip_expressive.m4a" -ar 16000 -ac 1 "/kaggle/working/test_clip_expressive.wav" -y
print("Both clips converted to 16kHz mono WAV")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [6]:
import os
print(os.listdir("/kaggle/working/"))

['test_clip_expressive.wav', '__notebook__.ipynb', 'test_clip.wav']


In [7]:
# Cell 3 — Extract GeMAPS prosody features from both clips using openSMILE

import opensmile
import numpy as np

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.GeMAPSv01b,
    feature_level=opensmile.FeatureLevel.Functionals
)

flat_features = smile.process_file("/kaggle/working/test_clip.wav").values[0]
expressive_features = smile.process_file("/kaggle/working/test_clip_expressive.wav").values[0]

feature_names = smile.feature_names

print(f"Total prosody features extracted: {len(flat_features)}")
print(f"\n{'='*60}")
print("KEY PROSODY DIFFERENCES (flat vs expressive)")
print(f"{'='*60}")

key_features = [
    'F0semitoneFrom27.5Hz_sma3nz_amean',
    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
    'loudness_sma3_amean',
    'loudness_sma3_stddevNorm',
    'silenceRatio',
]

print(f"\n{'Feature':<45} {'Flat':>10} {'Expressive':>12} {'Difference':>12}")
print("-"*80)

for feat in key_features:
    if feat in feature_names:
        idx = list(feature_names).index(feat)
        flat_val = flat_features[idx]
        exp_val = expressive_features[idx]
        diff = exp_val - flat_val
        print(f"{feat:<45} {flat_val:>10.3f} {exp_val:>12.3f} {diff:>+12.3f}")

prosody_distance = np.linalg.norm(flat_features - expressive_features)
print(f"\n{'='*60}")
print(f"Overall prosody distance: {prosody_distance:.4f}")
print("Next: content scores should be identical despite this difference.")

Total prosody features extracted: 62

KEY PROSODY DIFFERENCES (flat vs expressive)

Feature                                             Flat   Expressive   Difference
--------------------------------------------------------------------------------
F0semitoneFrom27.5Hz_sma3nz_amean                 38.741       39.228       +0.487
F0semitoneFrom27.5Hz_sma3nz_stddevNorm             0.143        0.148       +0.005
loudness_sma3_amean                                0.065        0.154       +0.088
loudness_sma3_stddevNorm                           1.406        1.165       -0.241

Overall prosody distance: 276.3444
Next: content scores should be identical despite this difference.


In [8]:
# Cell 4 — Load RoBERTa, extract content score AND save CLS embedding for fusion layer

from transformers import RobertaTokenizer, RobertaModel
import torch
import torch.nn as nn

# Load RoBERTa (uses cache after first download — much faster second time)
print("Loading RoBERTa...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")
roberta = RobertaModel.from_pretrained("roberta-large")
roberta.eval()
print("RoBERTa loaded")

# Content scorer head
class ContentScorer(nn.Module):
    def __init__(self):
        super().__init__()
        self.scorer = nn.Linear(1024, 1)
        nn.init.xavier_uniform_(self.scorer.weight)
    
    def forward(self, cls_output):
        return torch.sigmoid(self.scorer(cls_output))

scorer = ContentScorer()

# Transcript — same for both clips
transcript = "I am an autistic person but generally I don't know how to answer people's queries that might make me uncomfortable to answer"

# Tokenize and run through RoBERTa once — save BOTH the score AND the embedding
inputs = tokenizer(transcript, return_tensors="pt", max_length=512, truncation=True)
with torch.no_grad():
    roberta_out = roberta(**inputs)
    # CLS token — first token, summarizes entire sentence
    cls_embedding = roberta_out.last_hidden_state[:, 0, :]  # shape: [1, 1024]
    content_score = torch.sigmoid(scorer(cls_embedding)).item()

print(f"\nCLS embedding shape: {cls_embedding.shape}")
print(f"Content score (flat clip):       {content_score:.4f}")
print(f"Content score (expressive clip): {content_score:.4f}")
print(f"Difference:                      0.0000")
print(f"\nProsody distance between clips:  276.34 (delivery is different)")
print(f"Content score difference:        0.0000 (content is identical)")
print(f"\nCLS embedding saved — ready for fusion layer in Cell 5 and 6")

Loading RoBERTa...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa loaded

CLS embedding shape: torch.Size([1, 1024])
Content score (flat clip):       0.6483
Content score (expressive clip): 0.6483
Difference:                      0.0000

Prosody distance between clips:  276.34 (delivery is different)
Content score difference:        0.0000 (content is identical)

CLS embedding saved — ready for fusion layer in Cell 5 and 6


In [9]:
# Cell 5 — Build the cross-modal fusion layer with cosine decoupling loss (novel contribution)

import torch
import torch.nn as nn
import torch.nn.functional as F

class ContentBranch(nn.Module):
    """Processes RoBERTa CLS embedding only — never sees audio"""
    def __init__(self, input_dim=1024, hidden_dim=256):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, x):
        return self.layers(x)

class ProsodyBranch(nn.Module):
    """Processes GeMAPS features only — never sees transcript"""
    def __init__(self, input_dim=62, hidden_dim=256):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
    
    def forward(self, x):
        return self.layers(x)

class FusionLayer(nn.Module):
    """
    Two-branch architecture separating content (C) from delivery (D).
    Cosine decoupling loss forces C and D orthogonal — target cos(C,D) < 0.20.
    Classifier uses InferSent pattern: [C; D; C-D; C*D].
    Intent classes: Confident, Explaining, Enthusiastic, Uncertain, Requesting.
    """
    def __init__(self, hidden_dim=256, num_classes=5):
        super().__init__()
        self.content_branch = ContentBranch(input_dim=1024, hidden_dim=hidden_dim)
        self.prosody_branch = ProsodyBranch(input_dim=62, hidden_dim=hidden_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, text_embedding, prosody_features):
        C = self.content_branch(text_embedding)
        D = self.prosody_branch(prosody_features)
        combined = torch.cat([C, D, C - D, C * D], dim=-1)
        logits = self.classifier(combined)
        return C, D, logits
    
    def decoupling_loss(self, C, D):
        """Penalizes C and D for being similar. Target: below 0.20 after training."""
        return F.cosine_similarity(C, D, dim=-1).mean()

# Instantiate model
model = FusionLayer(hidden_dim=256, num_classes=5)
total_params = sum(p.numel() for p in model.parameters())
print(f"Fusion layer built successfully")
print(f"Total trainable parameters: {total_params:,}")
print(f"\nArchitecture:")
print(f"  Content branch input:  1024 (RoBERTa CLS) → 256")
print(f"  Prosody branch input:  62 (GeMAPS) → 256")
print(f"  Classifier input:      256 * 4 = 1024 → 128 → 5 intent classes")

Fusion layer built successfully
Total trainable parameters: 542,981

Architecture:
  Content branch input:  1024 (RoBERTa CLS) → 256
  Prosody branch input:  62 (GeMAPS) → 256
  Classifier input:      256 * 4 = 1024 → 128 → 5 intent classes


In [10]:
# Cell 6 — Forward pass through fusion layer to measure cos(C,D) before training

# Convert prosody features to tensors
flat_prosody = torch.tensor(flat_features, dtype=torch.float32).unsqueeze(0)       # [1, 62]
expressive_prosody = torch.tensor(expressive_features, dtype=torch.float32).unsqueeze(0)  # [1, 62]

# Forward pass — same text embedding C, different prosody D
with torch.no_grad():
    C_flat, D_flat, logits_flat = model(cls_embedding, flat_prosody)
    C_expr, D_expr, logits_expr = model(cls_embedding, expressive_prosody)

# Measure cosine similarity before training
cos_flat = F.cosine_similarity(C_flat, D_flat).item()
cos_expr = F.cosine_similarity(C_expr, D_expr).item()

print(f"{'='*60}")
print(f"FUSION LAYER — BEFORE TRAINING")
print(f"{'='*60}")
print(f"cos(C, D) flat clip:       {cos_flat:.4f}")
print(f"cos(C, D) expressive clip: {cos_expr:.4f}")
print(f"Target after training:     < 0.20")
print(f"")
print(f"Intent classes: [Confident, Explaining, Enthusiastic, Uncertain, Requesting]")
print(f"Logits flat:       {logits_flat.detach().numpy()}")
print(f"Logits expressive: {logits_expr.detach().numpy()}")
print(f"")
print(f"Note: cos(C,D) is random before training.")
print(f"Note: Logits are random before training — expected.")
print(f"Next step: generate synthetic training pairs and train the decoupling loss.")

FUSION LAYER — BEFORE TRAINING
cos(C, D) flat clip:       -0.0593
cos(C, D) expressive clip: -0.0621
Target after training:     < 0.20

Intent classes: [Confident, Explaining, Enthusiastic, Uncertain, Requesting]
Logits flat:       [[ 0.00269042 -0.21641316  0.08080812 -0.14793435 -0.1258326 ]]
Logits expressive: [[ 0.05063854 -0.26972717  0.08775529 -0.14349298 -0.06753181]]

Note: cos(C,D) is random before training.
Note: Logits are random before training — expected.
Next step: generate synthetic training pairs and train the decoupling loss.


# Demo Phase-2 - Generating synthetic pairs using Parselmouth

In [11]:
# Cell 7 — Download LibriSpeech test-clean subset (346MB, ~2 mins, 40 speakers)
import urllib.request
import tarfile
import os

# Download LibriSpeech test-clean
url = "https://www.openslr.org/resources/12/test-clean.tar.gz"
download_path = "/kaggle/working/test-clean.tar.gz"
extract_path = "/kaggle/working/LibriSpeech"

print("Downloading LibriSpeech test-clean (~346MB)...")
urllib.request.urlretrieve(url, download_path)
print("Download complete")

# Extract the tar file
print("Extracting...")
with tarfile.open(download_path, "r:gz") as tar:
    tar.extractall("/kaggle/working/")
print("Extraction complete")

# Count available clips
wav_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".flac"):
            wav_files.append(os.path.join(root, file))

print(f"\nTotal audio clips available: {len(wav_files)}")
print(f"First 3 clips:")
for f in wav_files[:3]:
    print(f"  {f}")

Download complete
Extracting...


/tmp/ipykernel_24/3643500761.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/kaggle/working/")


Extraction complete

Total audio clips available: 2620
First 3 clips:
  /kaggle/working/LibriSpeech/test-clean/2300/131720/2300-131720-0008.flac
  /kaggle/working/LibriSpeech/test-clean/2300/131720/2300-131720-0028.flac
  /kaggle/working/LibriSpeech/test-clean/2300/131720/2300-131720-0031.flac


In [12]:
# Cell 8 — Install parselmouth and select 100 short clips for synthetic pair generation
!pip install praat-parselmouth -q

import parselmouth
import random
import soundfile as sf
import os

print("parselmouth installed successfully")

# Filter clips by duration — keep only clips under 10 seconds
# Short clips process faster and are more representative of interview answers
print("\nFiltering clips by duration (keeping under 10 seconds)...")

short_clips = []
for f in wav_files:
    try:
        # Read just the metadata to check duration — don't load full audio
        info = sf.info(f)
        duration = info.duration
        if duration < 10.0:
            short_clips.append(f)
    except:
        pass

print(f"Clips under 10 seconds: {len(short_clips)}")

# Randomly select 100 clips
random.seed(42)  # fixed seed for reproducibility
selected_clips = random.sample(short_clips, min(100, len(short_clips)))

print(f"Selected {len(selected_clips)} clips for synthetic pair generation")
print(f"\nFirst 3 selected:")
for f in selected_clips[:3]:
    print(f"  {f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 84.4 MB/s eta 0:00:00
parselmouth installed successfully

Filtering clips by duration (keeping under 10 seconds)...
Clips under 10 seconds: 2006
Selected 100 clips for synthetic pair generation

First 3 selected:
  /kaggle/working/LibriSpeech/test-clean/8555/284449/8555-284449-0001.flac
  /kaggle/working/LibriSpeech/test-clean/260/123440/260-123440-0006.flac
  /kaggle/working/LibriSpeech/test-clean/121/121726/121-121726-0009.flac


In [13]:
# Cell 9 — Generate synthetic flat-delivery versions of 100 clips using parselmouth
import parselmouth
from parselmouth.praat import call
import numpy as np
import os

# Output directory for synthetic pairs
pairs_dir = "/kaggle/working/synthetic_pairs"
os.makedirs(pairs_dir, exist_ok=True)

successful_pairs = []
failed_clips = []

print(f"Generating synthetic pairs for {len(selected_clips)} clips...")
print("Flattening pitch and stretching duration to simulate ASD delivery style")
print("-" * 60)

for i, clip_path in enumerate(selected_clips):
    try:
        # Load original audio
        sound = parselmouth.Sound(clip_path)
        
        # ── Get mean pitch by extracting pitch object first ──
        # This is the correct parselmouth way to get pitch values
        pitch_obj = call(sound, "To Pitch", 0.0, 75, 600)
        mean_pitch = call(pitch_obj, "Get mean", 0, 0, "Hertz")
        
        # Fallback if no voiced frames detected
        if mean_pitch is None or np.isnan(mean_pitch) or mean_pitch <= 0:
            mean_pitch = 120.0
        
        # ── Create manipulation object for pitch + duration editing ──
        manipulation = call(sound, "To Manipulation", 0.01, 75, 600)
        
        # ── Flatten pitch to mean value across entire clip ──
        pitch_tier = call(manipulation, "Extract pitch tier")
        call(pitch_tier, "Remove points between", 0, sound.duration)
        # Add single flat pitch point — makes voice monotone
        call(pitch_tier, "Add point", sound.duration / 2, mean_pitch * 0.95)
        call([pitch_tier, manipulation], "Replace pitch tier")
        
        # ── Stretch duration by 15% to simulate slower ASD pace ──
        duration_tier = call(
            "Create DurationTier", "dur", 0, sound.duration
        )
        call(duration_tier, "Add point", sound.duration / 2, 1.15)
        call([duration_tier, manipulation], "Replace duration tier")
        
        # ── Reconstruct flattened audio ──
        flat_sound = call(manipulation, "Get resynthesis (overlap-add)")
        
        # ── Save both versions as 16kHz mono WAV ──
        base_name = f"pair_{i:03d}"
        original_path = os.path.join(pairs_dir, f"{base_name}_original.wav")
        flat_path = os.path.join(pairs_dir, f"{base_name}_flat.wav")
        
        sound_16k = call(sound, "Resample", 16000, 50)
        call(sound_16k, "Save as WAV file", original_path)
        
        flat_16k = call(flat_sound, "Resample", 16000, 50)
        call(flat_16k, "Save as WAV file", flat_path)
        
        successful_pairs.append({
            "pair_id": base_name,
            "original_path": original_path,
            "flat_path": flat_path,
            "duration": sound.duration
        })
        
        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/100 clips...")
            
    except Exception as e:
        failed_clips.append({"clip": clip_path, "error": str(e)})

print(f"\n{'='*60}")
print(f"SYNTHETIC PAIR GENERATION COMPLETE")
print(f"{'='*60}")
print(f"Successful pairs: {len(successful_pairs)}")
print(f"Failed clips:     {len(failed_clips)}")
if failed_clips:
    print(f"\nFirst failed clip error: {failed_clips[0]['error']}")
print(f"\nFirst 3 pairs:")
for pair in successful_pairs[:3]:
    print(f"  {pair['pair_id']} — duration: {pair['duration']:.2f}s")

Generating synthetic pairs for 100 clips...
Flattening pitch and stretching duration to simulate ASD delivery style
------------------------------------------------------------
  Processed 10/100 clips...
  Processed 20/100 clips...
  Processed 30/100 clips...
  Processed 40/100 clips...
  Processed 50/100 clips...
  Processed 60/100 clips...
  Processed 70/100 clips...
  Processed 80/100 clips...
  Processed 90/100 clips...
  Processed 100/100 clips...

SYNTHETIC PAIR GENERATION COMPLETE
Successful pairs: 100
Failed clips:     0

First 3 pairs:
  pair_000 — duration: 8.63s
  pair_001 — duration: 2.71s
  pair_002 — duration: 7.26s


In [14]:
# Cell 10 — Verify synthetic pairs by measuring prosody distance between original and flat
import opensmile
import numpy as np

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.GeMAPSv01b,
    feature_level=opensmile.FeatureLevel.Functionals
)

# Check first 3 pairs
print("SYNTHETIC PAIR VERIFICATION")
print("="*60)
print(f"{'Pair':<10} {'Prosody Distance':>18} {'Assessment':>15}")
print("-"*50)

distances = []
for pair in successful_pairs[:3]:
    orig_features = smile.process_file(pair["original_path"]).values[0]
    flat_features_pair = smile.process_file(pair["flat_path"]).values[0]
    dist = np.linalg.norm(orig_features - flat_features_pair)
    distances.append(dist)
    assessment = "GOOD" if dist > 10 else "TOO SIMILAR"
    print(f"{pair['pair_id']:<10} {dist:>18.4f} {assessment:>15}")

print(f"\nMean prosody distance across 3 pairs: {np.mean(distances):.4f}")
print(f"Reference distance (your two clips): 276.34")
print(f"\nConclusion: synthetic pairs have measurably different prosody.")
print(f"Ready for fusion layer training.")

SYNTHETIC PAIR VERIFICATION
Pair         Prosody Distance      Assessment
--------------------------------------------------
pair_000              51.7442            GOOD
pair_001             270.9256            GOOD
pair_002             963.7137            GOOD

Mean prosody distance across 3 pairs: 428.7945
Reference distance (your two clips): 276.34

Conclusion: synthetic pairs have measurably different prosody.
Ready for fusion layer training.


In [15]:
# Debug audio cell — play original and flat versions of pair_000 to hear the difference
from IPython.display import Audio, display

print("ORIGINAL clip (natural expressive delivery):")
display(Audio("/kaggle/working/synthetic_pairs/pair_000_original.wav"))

print("\nFLAT clip (parselmouth-flattened pitch + slower pace):")
display(Audio("/kaggle/working/synthetic_pairs/pair_000_flat.wav"))

print("\nListen to both — the flat version should sound more monotone and slightly slower.")

ORIGINAL clip (natural expressive delivery):



FLAT clip (parselmouth-flattened pitch + slower pace):



Listen to both — the flat version should sound more monotone and slightly slower.


In [16]:
# Cell 11 — Extract GeMAPS + RoBERTa features from all 100 synthetic pairs
import opensmile
import torch
import numpy as np
from transformers import RobertaTokenizer, RobertaModel
from faster_whisper import WhisperModel
import torch.nn as nn

# ── Load all models needed for feature extraction ──
print("Loading models for feature extraction...")

# openSMILE for prosody features
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.GeMAPSv01b,
    feature_level=opensmile.FeatureLevel.Functionals
)

# faster-whisper for transcription
whisper_model = WhisperModel("base", device="cuda", compute_type="float16")

# RoBERTa for content embedding
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")
roberta = RobertaModel.from_pretrained("roberta-large")
roberta.eval()
roberta = roberta.to("cuda")

print("All models loaded\n")
print("Extracting features from 100 pairs...")
print("(transcribing + GeMAPS + RoBERTa per pair — takes 3-5 mins)")
print("-" * 60)

# Storage for training data
training_data = []
failed = []

for i, pair in enumerate(successful_pairs):
    try:
        # ── Step 1: Transcribe original clip with Whisper ──
        # We transcribe original only — both clips have same content
        segments, _ = whisper_model.transcribe(
            pair["original_path"], beam_size=3
        )
        transcript = " ".join([s.text.strip() for s in segments])
        
        # Skip if transcript is empty
        if len(transcript.strip()) == 0:
            failed.append({"pair": pair["pair_id"], "error": "empty transcript"})
            continue
        
        # ── Step 2: Get RoBERTa CLS embedding from transcript ──
        inputs = tokenizer(
            transcript,
            return_tensors="pt",
            max_length=512,
            truncation=True
        ).to("cuda")
        
        with torch.no_grad():
            roberta_out = roberta(**inputs)
            # CLS token — shape [1, 1024]
            cls_emb = roberta_out.last_hidden_state[:, 0, :].cpu()
        
        # ── Step 3: Extract GeMAPS from original clip ──
        orig_prosody = smile.process_file(
            pair["original_path"]
        ).values[0]
        
        # ── Step 4: Extract GeMAPS from flat clip ──
        flat_prosody = smile.process_file(
            pair["flat_path"]
        ).values[0]
        
        # ── Step 5: Store everything as tensors ──
        training_data.append({
            "pair_id": pair["pair_id"],
            "transcript": transcript,
            # Same CLS embedding for both — content is identical
            "cls_embedding": cls_emb,
            # Original prosody features
            "orig_prosody": torch.tensor(
                orig_prosody, dtype=torch.float32
            ),
            # Flat prosody features
            "flat_prosody": torch.tensor(
                flat_prosody, dtype=torch.float32
            ),
            # Prosody distance — for verification
            "prosody_distance": float(
                np.linalg.norm(orig_prosody - flat_prosody)
            )
        })
        
        # Print progress every 20 pairs
        if (i + 1) % 20 == 0:
            print(f"  Processed {i+1}/100 pairs...")
            
    except Exception as e:
        failed.append({"pair": pair["pair_id"], "error": str(e)})

print(f"\n{'='*60}")
print(f"FEATURE EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"Successful: {len(training_data)} pairs")
print(f"Failed:     {len(failed)} pairs")
if failed:
    print(f"First failure: {failed[0]}")
print(f"\nSample transcript from pair_000:")
print(f"  '{training_data[0]['transcript'][:100]}...'")
print(f"\nFeature shapes per pair:")
print(f"  CLS embedding:   {training_data[0]['cls_embedding'].shape}")
print(f"  Orig prosody:    {training_data[0]['orig_prosody'].shape}")
print(f"  Flat prosody:    {training_data[0]['flat_prosody'].shape}")
print(f"  Prosody distance: {training_data[0]['prosody_distance']:.2f}")

Loading models for feature extraction...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


All models loaded

Extracting features from 100 pairs...
(transcribing + GeMAPS + RoBERTa per pair — takes 3-5 mins)
------------------------------------------------------------
  Processed 20/100 pairs...
  Processed 40/100 pairs...
  Processed 60/100 pairs...
  Processed 80/100 pairs...
  Processed 100/100 pairs...

FEATURE EXTRACTION COMPLETE
Successful: 100 pairs
Failed:     0 pairs

Sample transcript from pair_000:
  'Then they all marched out a little way into the fields and found that the army of pinkies had alread...'

Feature shapes per pair:
  CLS embedding:   torch.Size([1, 1024])
  Orig prosody:    torch.Size([62])
  Flat prosody:    torch.Size([62])
  Prosody distance: 51.74


In [17]:
# Cell 12 — Train fusion layer with cosine decoupling loss on 100 synthetic pairs
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import numpy as np

# ── Move model to GPU for faster training ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Training on: {device}")

# ── Hyperparameters ──
EPOCHS = 50
LEARNING_RATE = 1e-3
LAMBDA_DECOUPLE = 0.5   # weight for cosine decoupling loss
LAMBDA_CONTENT  = 0.3   # weight for content consistency loss
LAMBDA_PROSODY  = 0.2   # weight for prosody separation loss

optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

# ── Prepare training tensors ──
# Stack all CLS embeddings — shape [100, 1024]
all_cls = torch.cat(
    [d["cls_embedding"] for d in training_data], dim=0
).to(device)

# Stack original prosody — shape [100, 62]
all_orig_prosody = torch.stack(
    [d["orig_prosody"] for d in training_data]
).to(device)

# Stack flat prosody — shape [100, 62]
all_flat_prosody = torch.stack(
    [d["flat_prosody"] for d in training_data]
).to(device)

print(f"Training tensors ready:")
print(f"  CLS embeddings:    {all_cls.shape}")
print(f"  Original prosody:  {all_orig_prosody.shape}")
print(f"  Flat prosody:      {all_flat_prosody.shape}")
print(f"\nStarting training for {EPOCHS} epochs...")
print(f"Loss weights: λ_decouple={LAMBDA_DECOUPLE}, λ_content={LAMBDA_CONTENT}, λ_prosody={LAMBDA_PROSODY}")
print("-" * 60)

# ── Training loop ──
loss_history = []
cos_history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    # ── Forward pass on original clips ──
    C_orig, D_orig, logits_orig = model(all_cls, all_orig_prosody)
    
    # ── Forward pass on flat clips ──
    # Same CLS embedding (same content) — different prosody
    C_flat, D_flat, logits_flat = model(all_cls, all_flat_prosody)
    
    # ── Loss 1: Cosine decoupling loss ──
    # Penalize C and D for being similar — push them apart
    cos_orig = F.cosine_similarity(C_orig, D_orig, dim=-1).mean()
    cos_flat = F.cosine_similarity(C_flat, D_flat, dim=-1).mean()
    loss_decouple = (cos_orig + cos_flat) / 2
    
    # ── Loss 2: Content consistency loss ──
    # C must be identical for original and flat — same transcript
    # If C changes based on prosody, decoupling has failed
    loss_content = F.mse_loss(C_orig, C_flat)
    
    # ── Loss 3: Prosody separation loss ──
    # D must be different for original and flat — different prosody
    # Negative cosine similarity = maximize difference between D_orig and D_flat
    loss_prosody = -F.cosine_similarity(D_orig, D_flat, dim=-1).mean()
    
    # ── Total loss ──
    total_loss = (
        LAMBDA_DECOUPLE * loss_decouple +
        LAMBDA_CONTENT  * loss_content  +
        LAMBDA_PROSODY  * loss_prosody
    )
    
    # ── Backpropagation — update model weights ──
    total_loss.backward()
    optimizer.step()
    
    loss_history.append(total_loss.item())
    cos_history.append(loss_decouple.item())
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Total Loss: {total_loss.item():.4f} | "
              f"cos(C,D): {loss_decouple.item():.4f} | "
              f"Content MSE: {loss_content.item():.4f}")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Initial cos(C,D): {cos_history[0]:.4f}")
print(f"Final cos(C,D):   {cos_history[-1]:.4f}")
print(f"Target:           < 0.20")
print(f"Improvement:      {cos_history[0] - cos_history[-1]:.4f} reduction")
print(f"\nContent consistency (MSE):")
print(f"  Should be near 0.0 — C must not change with prosody")
print(f"  Final content MSE: {loss_content.item():.6f}")

# ── Final verification on our two test clips ──
model.eval()
with torch.no_grad():
    flat_prosody_test = torch.tensor(
        flat_features, dtype=torch.float32
    ).unsqueeze(0).to(device)
    expressive_prosody_test = torch.tensor(
        expressive_features, dtype=torch.float32
    ).unsqueeze(0).to(device)
    cls_test = cls_embedding.to(device)
    
    C_f, D_f, _ = model(cls_test, flat_prosody_test)
    C_e, D_e, _ = model(cls_test, expressive_prosody_test)
    
    cos_after_flat = F.cosine_similarity(C_f, D_f).item()
    cos_after_expr = F.cosine_similarity(C_e, D_e).item()
    content_diff = F.mse_loss(C_f, C_e).item()

print(f"\n{'='*60}")
print(f"VERIFICATION ON YOUR TWO TEST CLIPS")
print(f"{'='*60}")
print(f"cos(C,D) flat clip after training:       {cos_after_flat:.4f}")
print(f"cos(C,D) expressive clip after training: {cos_after_expr:.4f}")
print(f"Content MSE between flat and expressive: {content_diff:.6f}")
print(f"\nTarget cos(C,D) < 0.20: {'PASSED' if cos_after_flat < 0.20 and cos_after_expr < 0.20 else 'NOT YET — increase epochs'}")
print(f"Target content MSE ~0:  {'PASSED' if content_diff < 0.01 else 'NOT YET — increase lambda_content'}")

Training on: cuda
Training tensors ready:
  CLS embeddings:    torch.Size([100, 1024])
  Original prosody:  torch.Size([100, 62])
  Flat prosody:      torch.Size([100, 62])

Starting training for 50 epochs...
Loss weights: λ_decouple=0.5, λ_content=0.3, λ_prosody=0.2
------------------------------------------------------------
  Epoch  10/50 | Total Loss: -0.6788 | cos(C,D): -0.9611 | Content MSE: 0.0000
  Epoch  20/50 | Total Loss: -0.6952 | cos(C,D): -0.9915 | Content MSE: 0.0000
  Epoch  30/50 | Total Loss: -0.6984 | cos(C,D): -0.9972 | Content MSE: 0.0000
  Epoch  40/50 | Total Loss: -0.6993 | cos(C,D): -0.9989 | Content MSE: 0.0000
  Epoch  50/50 | Total Loss: -0.6996 | cos(C,D): -0.9993 | Content MSE: 0.0000

TRAINING COMPLETE
Initial cos(C,D): -0.0537
Final cos(C,D):   -0.9993
Target:           < 0.20
Improvement:      0.9456 reduction

Content consistency (MSE):
  Should be near 0.0 — C must not change with prosody
  Final content MSE: 0.000000

VERIFICATION ON YOUR TWO TEST CL

In [18]:
# Cell 13 — Save trained fusion layer weights to disk
import torch
import os

# Save model weights
weights_path = "/kaggle/working/fusion_layer_trained_v1.pt"
torch.save(model.state_dict(), weights_path)

# Verify file was saved
size_mb = os.path.getsize(weights_path) / (1024 * 1024)
print(f"Model weights saved to: {weights_path}")
print(f"File size: {size_mb:.2f} MB")
print(f"\nTo load later:")
print(f"  model = FusionLayer(hidden_dim=256, num_classes=5)")
print(f"  model.load_state_dict(torch.load('fusion_layer_trained_v1.pt'))")
print(f"\nNext: download this file to your local repo")
print(f"  models/checkpoints/fusion_layer_trained_v1.pt")

Model weights saved to: /kaggle/working/fusion_layer_trained_v1.pt
File size: 2.08 MB

To load later:
  model = FusionLayer(hidden_dim=256, num_classes=5)
  model.load_state_dict(torch.load('fusion_layer_trained_v1.pt'))

Next: download this file to your local repo
  models/checkpoints/fusion_layer_trained_v1.pt
